# Pertussis Sanity Check
https://www.cdc.gov/pertussis/php/surveillance/index.html?CDC_AA_refVal=https%3A%2F%2Fwww.cdc.gov%2Fpertussis%2Fphp%2Fsurveillance%2Fpertussis-incidence-by-age-group-and-year.html

In [1]:
import pandas as pd

pertussis_df = pd.read_csv('../app/data/pertussis_age.csv')
ms_df = pd.read_csv('../app/data/cases_coverage_ms.csv')

In [2]:
ms_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated,population,measles_cases_per_100k,mumps_cases_per_100k,pertussis_cases_per_100k
0,1995,AK,NaN,13.0,1.0,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0,604412.0,NaN,2.150851,0.165450
1,1996,AK,63.0,3.0,4.0,84.55,260.0,225.0,84.55,260.0,225.0,78.25,260.0,210.0,608569.0,10.352154,0.492960,0.657280
2,1997,AK,48.0,4.0,14.0,87.41,291.0,257.0,87.41,291.0,257.0,80.98,291.0,242.0,612968.0,7.830751,0.652563,2.283969
3,1998,AK,29.0,2.0,15.0,87.06,34.0,30.0,87.06,34.0,30.0,82.01,34.0,28.0,619932.0,4.677932,0.322616,2.419620
4,1999,AK,29.0,3.0,5.0,90.67,349.0,321.0,90.67,349.0,321.0,83.54,349.0,296.0,624779.0,4.641641,0.480170,0.800283


In [8]:
avg_vacc_coverage_df = ms_df[['year', 'pertussis_coverage_pct']]
avg_vacc_coverage_df = avg_vacc_coverage_df.groupby('year').mean()
len(avg_vacc_coverage_df)

23

In [10]:
pertussis_df.head()

,Year,<1 yr,1-6 yrs,7-10 yrs,11-19 yrs,20+ yrs,Overall Incidence,Data Status
0,1990,47.89,5.57,1.68,1.38,0.19,1.83,Finalized
1,1991,27.27,3.20,0.96,0.58,0.14,1.07,Finalized
2,1992,38.41,3.93,1.95,1.99,0.27,1.59,Finalized
3,1993,68.00,6.59,3.26,2.08,0.33,2.53,Finalized
4,1994,42.61,4.35,2.26,2.30,0.27,1.75,Finalized


In [ ]:
child_pertussis_cases_df = pertussis_df[['Year', '<1 yr', '1-6 yrs']]
child_pertussis_cases_df.rename(columns={'Year': 'year'}, inplace=True)
child_pertussis_cases_df['0-6 yrs'] = child_pertussis_cases_df['<1 yr'] + child_pertussis_cases_df['1-6 yrs']
child_pertussis_cases_df.head()

,year,<1 yr,1-6 yrs,0-6 yrs
0,1990,47.89,5.57,53.46
1,1991,27.27,3.20,30.47
2,1992,38.41,3.93,42.34
3,1993,68.00,6.59,74.59
4,1994,42.61,4.35,46.96


In [16]:
sanity_df = pd.merge(avg_vacc_coverage_df, child_pertussis_cases_df, on='year')
len(sanity_df)

23

In [17]:
sanity_df.head()

,year,pertussis_coverage_pct,<1 yr,1-6 yrs,0-6 yrs
0,1995,79.347551,48.74,4.71,53.45
1,1996,81.532600,63.23,6.01,69.24
2,1997,82.680000,52.73,4.49,57.22
3,1998,84.052000,56.71,5.45,62.16
4,1999,84.413400,57.12,4.57,61.69


In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def graph_pertussis_age_coverage(df):

    fig = make_subplots(
        specs=[[{"secondary_y": True}]]
    )

    colors = {
        "<1 yr": "#E74C3C",
        "1-6 yrs": "#F39C12",
        "0-6 yrs": "#9B59B6"
    }

    # Three grouped bars per year
    for age_group, color in colors.items():

        fig.add_trace(
            go.Bar(
                x=df["year"],
                y=df[age_group],
                name=age_group,
                marker_color=color,
                hovertemplate=(
                    "Year: %{x}<br>"
                    f"Age group: {age_group}<br>"
                    "Incidence: %{y:.2f}"
                    "<extra></extra>"
                )
            ),
            secondary_y=False
        )

    # Vaccination coverage line
    fig.add_trace(
        go.Scatter(
            x=df["year"],
            y=df["pertussis_coverage_pct"],
            name="DTaP Coverage (%)",
            mode="lines+markers",
            line=dict(color="#3498DB", width=3),
            marker=dict(size=7),
            hovertemplate=(
                "Year: %{x}<br>"
                "Coverage: %{y:.2f}%"
                "<extra></extra>"
            )
        ),
        secondary_y=True
    )

    fig.update_yaxes(
        title_text="Pertussis Incidence (Cases per 100K)",
        secondary_y=False
    )

    fig.update_yaxes(
        title_text="Vaccination Coverage (%)",
        range=[0, 100],
        secondary_y=True
    )

    fig.update_xaxes(
        title_text="Year",
        dtick=1
    )

    fig.update_layout(
        title="National Pertussis Incidence by Age Group & DTaP Coverage",
        barmode="group",
        template="plotly_white",
        height=550,
        hovermode="x unified",
        legend=dict(orientation="h", y=1.12, x=0),
        margin=dict(t=110)
    )

    return fig

In [20]:
fig = graph_pertussis_age_coverage(sanity_df)
fig.show()

In [24]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

def pearson_pertussis_age(df, start_year=None, end_year=None):

    age_groups = ["<1 yr", "1-6 yrs", "0-6 yrs"]
    results = []

    # Filter by selected year range
    data = df.copy()

    if start_year is not None:
        data = data[data["year"] >= start_year]

    if end_year is not None:
        data = data[data["year"] <= end_year]

    for age_group in age_groups:

        # Keep years with both coverage and incidence
        valid = data[
            ["year", "pertussis_coverage_pct", age_group]
        ].dropna()

        x = valid["pertussis_coverage_pct"]
        y = valid[age_group]

        n = len(valid)

        if n >= 3 and x.nunique() > 1 and y.nunique() > 1:
            r, p = pearsonr(x, y)
        else:
            r, p = np.nan, np.nan

        results.append({
            "age_group": age_group,
            "start_year": start_year,
            "end_year": end_year,
            "n": n,
            "pearson_r": r,
            "p_value": p,
            "significant_0_05": bool(p < 0.05) if np.isfinite(p) else False
        })

    return pd.DataFrame(results)

In [26]:
# Full dataset
pearson_pertussis_age(sanity_df)

,age_group,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,None,None,23,0.269977,0.212819,False
1,1-6 yrs,None,None,23,0.297041,0.168693,False
2,0-6 yrs,None,None,23,0.282120,0.192156,False


In [ ]:
# Before the pertussis resurgence
pearson_pertussis_age(sanity_df, 1995, 2009)

,age_group,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,1995,2009,15,0.403509,0.135834,False
1,1-6 yrs,1995,2009,15,0.562551,0.029032,True
2,0-6 yrs,1995,2009,15,0.445562,0.096016,False


In [28]:
# Pertussis resurgence period
pearson_pertussis_age(sanity_df, 2010, 2014)

,age_group,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,2010,2014,5,-0.832937,0.079884,False
1,1-6 yrs,2010,2014,5,-0.751247,0.143245,False
2,0-6 yrs,2010,2014,5,-0.816459,0.091749,False


In [29]:
# After the resurgence
pearson_pertussis_age(sanity_df, 2015, 2024)

,age_group,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,2015,2024,3,0.772482,0.438030,False
1,1-6 yrs,2015,2024,3,0.426067,0.719797,False
2,0-6 yrs,2015,2024,3,0.741907,0.467843,False


In [30]:
# Exclude 2010–2014 entirely
pearson_pertussis_age(
    sanity_df[~sanity_df["year"].between(2010, 2014)]
)

,age_group,start_year,end_year,n,pearson_r,p_value,significant_0_05
0,<1 yr,None,None,18,0.412341,0.089047,False
1,1-6 yrs,None,None,18,0.488415,0.039728,True
2,0-6 yrs,None,None,18,0.463597,0.052651,False
